# Генерация текстовых данных через β-VAE

**Пайплайн:** `dataset` → `SentenceTransformer (all-MiniLM-L6-v2)` → `β-VAE` → синтетические эмбеддинги → seq2seq декодирование → валидация

Вся реализация вынесена в пакет `synthetic_vae/`. Ноутбук отвечает только за конфигурацию и демонстрацию результатов.

In [ ]:
%pip install sentence-transformers torch transformers sentencepiece scipy scikit-learn --quiet

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from synthetic_vae import SyntheticEmbeddingPipeline

pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_columns', None)
print('import success.')

## Предобработка данных

In [ ]:
RAW_PATH     = '../data/test.csv'
N_SAMPLES    = 10000
RANDOM_STATE = 42

df_raw = pd.read_csv(RAW_PATH)
df = (
    df_raw.iloc[:, 1:]
    .drop_duplicates(keep='first')
    .dropna()
    .sample(n=N_SAMPLES, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print(f'df.shape: {df.shape}')
df.head(2)

## Настройка гиперпараметров пайплайна

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

pipeline = SyntheticEmbeddingPipeline(
    model_name       = 'sentence-transformers/all-MiniLM-L6-v2',
    local_model_path = './models/all-MiniLM-L6-v2',
    hidden_dim       = 128,
    latent_dim       = 50,
    batch_size       = 128,
    epochs           = 100,
    lr               = 1e-4,
    num_synthetic    = 10000,
    base_beta        = 0.01,   # mock
    max_beta         = 0.08,
    warmup_ratio     = 0.4,
    device           = device,
    verbose          = True,
)
print(f'Device: {pipeline.device}')

## Запуск пайплайна

In [ ]:
pipeline.fit(df)

In [ ]:
import matplotlib.pyplot as plt


plt.style.use('darkgrid') #'seaborn-v0_8-darkgrid'

fig, axes = plt.subplots(1, len(pipeline.synthetic_data_), figsize=(7 * len(pipeline.synthetic_data_), 4))
if len(pipeline.synthetic_data_) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, pipeline.synthetic_data_.iterrows()):
    col          = row['column']
    loss_history = row['loss_history']
    epochs_range = range(1, len(loss_history) + 1)

    ax.fill_between(range(1, len(loss_history) + 1), loss_history, alpha=0.2, color='steelblue')
    ax.plot(range(1, len(loss_history) + 1), loss_history, color='steelblue', lw=2)
    ax.set_title(f"Loss '{col}'", fontsize=13)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Total Loss')
    ax.grid(True, alpha=0.3)

    min_epoch = int(np.argmin(loss_history)) + 1
    min_loss  = min(loss_history)
    ax.annotate(
        f'min={min_loss:.4f}\nepoch {min_epoch}',
        xy=(min_epoch, min_loss),
        xytext=(min_epoch + len(loss_history) * 0.05, min_loss + (max(loss_history) - min_loss) * 0.1),
        arrowprops=dict(arrowstyle='->', color='tomato'),
        fontsize=9, color='tomato'
    )

plt.suptitle('β-VAE training loss (warmup β, τ=1.0)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Основные стат. моменты эмбеддингов оригинальных данных

In [ ]:
scalar_cols = ['column', 'n_entries', 'vector_dim',
               'overall_mean', 'overall_std',
               'overall_q25', 'overall_q50', 'overall_q75', 'overall_iqr']
pipeline.summary_df_[scalar_cols]

## Валидация

In [ ]:
pivot_df, detailed_df, wasserstein_df = pipeline.validate(n_wasserstein_projections=100)

In [ ]:
pivot_df

In [ ]:
wasserstein_df

In [ ]:
score_cols = [
    'column', 'composite_score', 'composite_score_with_swd',
    'sliced_wasserstein', 'normalized_swd',
    'centr_cos_sim', 'syn_diversity_avg', 'orig_diversity_avg',
    'rel_err_mean', 'rel_err_std'
]
detailed_df[score_cols]

In [ ]:
detailed_df.T

Синтетические эмбеддинги доступны как `dict[str, np.ndarray]` для дальнейшего использования.

In [ ]:
synthetic_embeddings = pipeline.get_synthetic_embeddings()

for col, emb in synthetic_embeddings.items():
    print(f'{col}: shape={emb.shape},  mean={emb.mean():.6f},  std={emb.std():.6f}')

## Обучение seq2seq декодера

Декодер обучается на парах *(оригинальный эмбеддинг, оригинальный текст)* из входного датасета.
Гиперпараметры (число эпох, `max_new_tokens`) выбираются автоматически по размеру выборки и длине текстов.

Базовая модель — **T5-small** (60M params), альтернативная — **google/t5-efficient-tiny** (4B params) для быстрой генерации.

In [ ]:
DECODE_COLS = None

pipeline.fit_decoder(
    df,
    text_cols          = DECODE_COLS,
    decoder_model_name = 'google/t5-efficient-tiny',
    decoder_epochs     = None,
    decoder_batch_size = 32,
    decoder_lr         = 3e-4,
)

In [ ]:
SHOW_COL  = 'highlights'
N_SHOW    = 2
NUM_BEAMS = 4

generated_texts = pipeline.decode_texts(SHOW_COL, n=N_SHOW, num_beams=NUM_BEAMS)
original_texts = df[SHOW_COL].tolist()[:N_SHOW]

st_model      = SentenceTransformer(pipeline.local_model_path)
syn_vecs_show = synthetic_embeddings[SHOW_COL][:N_SHOW]
gen_vecs      = st_model.encode(generated_texts, convert_to_numpy=True)
cos_scores    = [
    float(cosine_similarity(syn_vecs_show[i:i+1], gen_vecs[i:i+1])[0][0])
    for i in range(N_SHOW)
]

for i in range(N_SHOW):
    print(f'{i + 1}: {cos_scores[i]:.4f}')
    print(f'{original_texts[i][:400]}')
    print(f'\n {generated_texts[i]}')

In [ ]:
# all_synthetic_texts = {}
#
# for col in pipeline.decoders_.keys():
#     print(f"decode vec for '{col}'...")
#     texts = pipeline.decode_texts(col, num_beams=4)
#     all_synthetic_texts[col] = texts
#     print(f"  → generate {len(texts)} texts")
#
# synthetic_texts_df = pd.DataFrame(all_synthetic_texts)
# print(f'synthetic_texts_df.shape: {synthetic_texts_df.shape}')
# synthetic_texts_df.head(3)